C29. Таксономия отказов: о чём пишет судья, когда пункт не выполнен.....


In [1]:
import sys, os, json, re

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/evaluation_robustness')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF
from sklearn.metrics import silhouette_score
A = load_arms()
ARMS_T = ['P-base', 'baseline gpt', 'baseline deepseekv4', 'homog gpt', 'USC@7 gpt', 'P-strong']
ARMS_T = [a for a in ARMS_T if a in A]


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


In [2]:
def explanations(arm):
    d = A[arm]
    rows = []
    for v, s in d['judge_details'].dropna().items():
        try:
            arr = json.loads(s)
        except Exception:
            continue
        for it in arr:
            ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
            rows.append({'arm': arm, 'case': v, 'cid': it.get('id'), 'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax, 'text': str(it.get('explanation') or '')})
    return pd.DataFrame(rows)
EX = pd.concat([explanations(a) for a in ARMS_T], ignore_index=True)
EX = EX[EX['text'].str.len() > 10]
print('объяснений извлечено: %d  (плеч %d)' % (len(EX), EX['arm'].nunique()))
show(EX.groupby('arm').agg(всего=('met', 'size'), не_выполнено=('met', lambda s: int((~s).sum())), доля_отказов=('met', lambda s: float((~s).mean()))), 'Объём материала по плечам')
FAIL = EX[~EX['met']].copy()
print('\nтекстов об отказах: %d' % len(FAIL))
print('медианная длина объяснения: %d символов' % int(FAIL['text'].str.len().median()))


объяснений извлечено: 19599  (плеч 6)

Объём материала по плечам
------------------------------------------------------------
                     всего  не_выполнено  доля_отказов
arm                                                   
P-base                3269          1485        0.4543
P-strong              3269          1441        0.4408
USC@7 gpt             3269          1840        0.5629
baseline deepseekv4   3254          1815        0.5578
baseline gpt          3269          1831        0.5601
homog gpt             3269          1534        0.4693

текстов об отказах: 9946
медианная длина объяснения: 173 символов


In [3]:
vec = TfidfVectorizer(max_features=6000, ngram_range=(1, 2), min_df=5, max_df=0.6, stop_words='english', sublinear_tf=True)
X = vec.fit_transform(FAIL['text'])
terms = np.array(vec.get_feature_names_out())
print('матрица признаков:', X.shape)
rng = np.random.default_rng(0)
sub_idx = rng.choice(X.shape[0], min(4000, X.shape[0]), replace=False)
rows = []
for k in [3, 4, 5, 6, 7, 8]:
    km = KMeans(n_clusters=k, n_init=5, random_state=0).fit(X)
    sil = silhouette_score(X[sub_idx], km.labels_[sub_idx], metric='cosine')
    rows.append({'k': k, 'силуэт': sil, 'инерция': km.inertia_})
show(pd.DataFrame(rows).set_index('k'), 'Подбор числа кластеров')
K = int(pd.DataFrame(rows).set_index('k')['силуэт'].idxmax())
print('\nвыбрано k = %d (максимум силуэта)' % K)
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(X)
FAIL['кластер'] = km.labels_
centers = km.cluster_centers_
print()
for c in range(K):
    top = np.argsort(-centers[c])[:14]
    n = int((km.labels_ == c).sum())
    print('кластер %d  (n = %d, %.1f%%)' % (c, n, 100 * n / len(FAIL)))
    print('   ', ', '.join(terms[top]))


матрица признаков: (9946, 6000)


  File "D:\PythonUser\Python313\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")



Подбор числа кластеров
------------------------------------------------------------
   силуэт   инерция
k                  
3  0.0110 9728.9180
4  0.0140 9684.7500
5  0.0148 9652.0418
6  0.0146 9622.2068
7  0.0165 9593.7982
8  0.0176 9567.7329

выбрано k = 8 (максимум силуэта)



кластер 0  (n = 492, 4.9%)
    history, prior, protocol does, asking, mention asking, family, family history, inquire, does inquire, mention, recent, status, episodes, does mention
кластер 1  (n = 551, 5.5%)
    does recommend, recommend, protocol does, instead, antibiotics, therapy, uncomplicated, advises, topical, line, routine, suggests, imaging, workup
кластер 2  (n = 519, 5.2%)
    antibiotics, topical, prescribe, does prescribe, spectrum, corticosteroids, oral, broad, broad spectrum, recommends, systemic, protocol recommends, oral antibiotics, iv
кластер 3  (n = 3879, 39.0%)
    mention, does mention, protocol does, does explicitly, protocol includes, includes, patient, explicitly, protocol advises, counseling, advises, explicitly mention, risk, use
кластер 4  (n = 700, 7.0%)
    correctly, protocol correctly, correctly identifies, identifies, interprets, correctly interprets, based, findings, elevated, diagnoses, acute, indicative, correctly diagnoses, condition
кластер 5  (n =

In [4]:
for c in range(K):
    sub = FAIL[FAIL['кластер'] == c]
    d = np.asarray(X[FAIL['кластер'].to_numpy() == c].dot(centers[c].reshape(-1, 1))).ravel()
    order = np.argsort(-d)[:3]
    print('\n--- кластер %d ---' % c)
    for i in order:
        t = re.sub('\\s+', ' ', sub.iloc[i]['text'])[:240]
        print('   [%s] %s' % (sub.iloc[i]['axis'], t))



--- кластер 0 ---
   [uncertainty] The protocol does not mention asking about vaccination status, recent infections, family history of cardiovascular disease, or prior medical history.
   [uncertainty] The protocol does not mention asking about vaccination status, recent infections, family history of cardiovascular disease, or prior medical history.
   [uncertainty] The protocol does not mention asking about prior episodes, family history, or immunosuppression/diabetes risk factors.

--- кластер 1 ---
   [accuracy] The protocol does not recommend quinine or mention FDA warnings related to it.
   [accuracy] The protocol does not recommend the use of teething necklaces and instead suggests safe teething comfort measures.
   [accuracy] The protocol does not recommend teething necklaces and instead suggests safe non-pharmacologic comfort measures.

--- кластер 2 ---
   [accuracy] The protocol recommends oral antibiotics appropriate for outpatient management and does not prescribe broad-sp

In [5]:
nmf = NMF(n_components=K, random_state=0, init='nndsvda', max_iter=400)
Wm = nmf.fit_transform(X)
Hm = nmf.components_
for c in range(K):
    top = np.argsort(-Hm[c])[:14]
    print('тема %d: %s' % (c, ', '.join(terms[top])))
FAIL['тема NMF'] = Wm.argmax(axis=1)
print()
show(pd.crosstab(FAIL['кластер'], FAIL['тема NMF']), 'Согласие K-means и NMF (строки — кластеры, столбцы — темы)')
agree = float((FAIL['кластер'].to_numpy() == pd.factorize(FAIL['тема NMF'])[0]).mean())
print('\nдоля совпадений меток (без учёта перестановки): %.3f' % agree)
print('Точное совпадение не требуется: метки кластеров произвольны.')
print('Важно, что таблица сопряжённости близка к блочно-диагональной.')


тема 0: explicitly, protocol explicitly, includes, explicitly includes, advises, explicitly advises, does explicitly, explicitly recommends, explicitly states, states, protocol includes, addressing, risk, patient
тема 1: does mention, mention, protocol does, use, risk, monitoring, protocol advises, including, differential, protocol includes, testing, prevention, explicitly mention, options
тема 2: recommends, protocol recommends, therapy, antibiotics, iv, line, spectrum, oral, broad, broad spectrum, treatment, immediate, initiating, recommends initiating
тема 3: correctly, protocol correctly, identifies, correctly identifies, based, findings, interprets, correctly interprets, diagnoses, diagnosis, elevated, acute, differential, indicative
тема 4: care, emergent, appropriately, protocol appropriately, self, emergency, non, self care, condition, non emergent, urgent, emergent care, evaluation, management
тема 5: recommend, does recommend, protocol does, topical, corticosteroids, antibiot

In [6]:
dist = pd.crosstab(FAIL['arm'], FAIL['кластер'], normalize='index')
show(dist, 'Доля каждого типа отказа внутри плеча')
cnt = pd.crosstab(FAIL['arm'], FAIL['кластер'])
show(cnt, 'Абсолютные числа отказов по типам')
print()
print('Хи-квадрат на однородность распределения типов отказов между плечами:')
chi2, pv, dof, _ = stats.chi2_contingency(cnt)
print('   chi2 = %.1f, df = %d, p = %s' % (chi2, dof, fmt_p(pv)))
print()
if 'P-base' in cnt.index and 'baseline gpt' in cnt.index:
    rows = []
    for c in cnt.columns:
        a_, b_ = (int(cnt.loc['P-base', c]), int(cnt.loc['baseline gpt', c]))
        rows.append({'кластер': c, 'P-base': a_, 'baseline gpt': b_, 'сокращение, %': 100 * (1 - a_ / b_) if b_ else np.nan})
    R = pd.DataFrame(rows).set_index('кластер')
    R.loc['ВСЕГО'] = [cnt.loc['P-base'].sum(), cnt.loc['baseline gpt'].sum(), 100 * (1 - cnt.loc['P-base'].sum() / cnt.loc['baseline gpt'].sum())]
    show(R, 'Насколько панель сокращает каждый тип отказа')
    print()
    print('Типы, сокращённые СИЛЬНЕЕ среднего, — это то, что панель лечит.')
    print('Типы, сокращённые слабее, — её остаточные слабости.')



Доля каждого типа отказа внутри плеча
------------------------------------------------------------
кластер                  0      1      2      3      4      5      6      7
arm                                                                        
P-base              0.0458 0.0545 0.0593 0.3293 0.0727 0.1798 0.1865 0.0721
P-strong            0.0451 0.0604 0.0548 0.3164 0.0784 0.1721 0.2117 0.0611
USC@7 gpt           0.0554 0.0576 0.0489 0.4386 0.0658 0.1592 0.1152 0.0592
baseline deepseekv4 0.0540 0.0512 0.0479 0.4171 0.0716 0.1631 0.1438 0.0512
baseline gpt        0.0557 0.0552 0.0481 0.4468 0.0628 0.1693 0.1032 0.0590
homog gpt           0.0372 0.0541 0.0567 0.3598 0.0737 0.1819 0.1708 0.0658

Абсолютные числа отказов по типам
------------------------------------------------------------
кластер                0    1   2    3    4    5    6    7
arm                                                       
P-base                68   81  88  489  108  267  277  107
P-strong           

In [7]:
show(pd.crosstab(FAIL['кластер'], FAIL['axis'], normalize='index'), 'Распределение типов отказов по осям рубрики')
show(FAIL.groupby('кластер').agg(средний_вес_пункта=('points', 'mean'), наблюдений=('points', 'size')), 'Вес пунктов, на которых случается каждый тип отказа')
ITEMS, POS = load_items(verbose=False)
outside = [m for m in ITEMS if m.startswith('baseline ') and m.replace('baseline ', '') not in COMPOSITION['P-base']]
if len(outside) >= 2:
    diff = {k: 1 - np.mean([ITEMS[m][k]['met'] for m in outside]) for k in POS}
    FAIL['трудность'] = [diff.get((r.case, r.cid), np.nan) for r in FAIL.itertuples()]
    show(FAIL.groupby('кластер')['трудность'].agg(['mean', 'median', 'size']), 'Трудность пунктов по типам отказа')



Распределение типов отказов по осям рубрики
------------------------------------------------------------
axis     accuracy  communication_quality  completeness  reasoning  uncertainty  urgency
кластер                                                                                
0          0.0244                 0.0000        0.0691     0.0020       0.8984   0.0061
1          0.8022                 0.0018        0.0272     0.0036       0.0236   0.1416
2          0.7900                 0.0000        0.1484     0.0058       0.0000   0.0559
3          0.1436                 0.1763        0.4586     0.0673       0.1274   0.0268
4          0.7714                 0.0000        0.0000     0.1171       0.0086   0.1029
5          0.4087                 0.0301        0.2652     0.0154       0.0171   0.2634
6          0.2842                 0.0159        0.3991     0.0631       0.0936   0.1441
7          0.0743                 0.0314        0.1007     0.0017       0.0165   0.7756

Вес пунктов, 


Трудность пунктов по типам отказа
------------------------------------------------------------
          mean  median  size
кластер                     
0       0.9878  1.0000   492
1       0.7917  1.0000   551
2       0.8427  1.0000   519
3       0.8527  1.0000  3879
4       0.3571  0.5000   700
5       0.8044  1.0000  1693
6       0.7585  1.0000  1506
7       0.7214  0.7500   606


In [8]:
LEX = {'пропуск (omission)': '\\b(does not (mention|address|include|discuss|specify)|no mention|not mentioned|omits?|omitted|fails? to (mention|include|address)|absent|missing|lacks?)\\b', 'неверно / противоречие': '\\b(incorrect|wrong|contradict\\w*|inaccurate|erroneous|mis(state|identif|interpret)\\w*|conflicts?)\\b', 'недостаточно конкретно': '\\b(vague|generic|non-?specific|insufficient\\w*|too general|without (specif|detail)\\w*|does not (quantif|specif)\\w*)\\b', 'не по форме рубрики': '\\b(implicit\\w*|only indirectly|partially|does not explicitly|not explicit\\w*)\\b'}
for lab, pat in LEX.items():
    FAIL[lab] = FAIL['text'].str.contains(pat, case=False, regex=True, na=False)
lex_tab = FAIL.groupby('arm')[list(LEX)].mean()
lex_tab['ни одна категория'] = 1 - FAIL.groupby('arm')[list(LEX)].max(axis=None).mean() if False else np.nan
show(lex_tab, 'Доля отказов, попадающих в словарные категории')
print()
print('Категории не взаимоисключающие: один текст может содержать несколько')
print('маркеров. Строка «ни одна категория» опущена как неинформативная —')
print('вместо неё смотрите на покрытие ниже.')
cov = FAIL[list(LEX)].any(axis=1).mean()
print('покрытие словарём: %.3f отказов содержат хотя бы один маркер' % cov)
print()
if 'P-base' in lex_tab.index and 'baseline gpt' in lex_tab.index:
    cmp_ = pd.DataFrame({'P-base': lex_tab.loc['P-base'], 'baseline gpt': lex_tab.loc['baseline gpt']})
    cmp_['разница'] = cmp_['P-base'] - cmp_['baseline gpt']
    show(cmp_, 'Структура отказов: панель против одиночной модели')
    print()
    print('ВАЖНО: это доли ВНУТРИ отказов. У панели отказов меньше в принципе,')
    print('поэтому рост доли какой-то категории не означает роста её числа.')
    if 'P-base' in cnt.index:
        n_p, n_b = (int(cnt.loc['P-base'].sum()), int(cnt.loc['baseline gpt'].sum()))
        abs_cmp = pd.DataFrame({'P-base, штук': (lex_tab.loc['P-base'] * n_p).round(0), 'baseline gpt, штук': (lex_tab.loc['baseline gpt'] * n_b).round(0)})
        abs_cmp['сокращение, %'] = 100 * (1 - abs_cmp.iloc[:, 0] / abs_cmp.iloc[:, 1])
        show(abs_cmp, 'То же в абсолютных числах')



Доля отказов, попадающих в словарные категории
------------------------------------------------------------
                     пропуск (omission)  неверно / противоречие  недостаточно конкретно  не по форме рубрики  ни одна категория
arm                                                                                                                            
P-base                           0.2343                  0.0054                  0.0222               0.1421                NaN
P-strong                         0.2353                  0.0056                  0.0201               0.1242                NaN
USC@7 gpt                        0.3810                  0.0038                  0.0245               0.1152                NaN
baseline deepseekv4              0.4088                  0.0039                  0.0325               0.0915                NaN
baseline gpt                     0.3856                  0.0049                  0.0229               0.1174               

In [9]:
save(FAIL.drop(columns=['text']), 'rubric_failure_taxonomy_failures_labeled.csv')
save(dist.reset_index(), 'rubric_failure_taxonomy_type_distribution.csv')
save(lex_tab.reset_index(), 'rubric_failure_taxonomy_lexicon.csv')
FAIL[['arm', 'case', 'cid', 'axis', 'кластер', 'text']].head(400).to_csv(path_join(OUT, 'rubric_failure_taxonomy_examples.csv'), sep=';', encoding='utf-8-sig', index=False)
print('сохранено: out/rubric_failure_taxonomy_examples.csv (400 примеров для ручного чтения)')


сохранено: out/rubric_failure_taxonomy_failures_labeled.csv  (9946 строк)
сохранено: out/rubric_failure_taxonomy_type_distribution.csv  (6 строк)
сохранено: out/rubric_failure_taxonomy_lexicon.csv  (6 строк)
сохранено: out/rubric_failure_taxonomy_examples.csv (400 примеров для ручного чтения)
